## Honey-Bee Hive Health Prediction: Baseline modeling

**Info:** In this notebook, we will implement a simple supervised model (Logistic Regression) on our entire training dataset containing Honey-Bee health descriptors and local weather features to predict weather a hive in a given apiary is healthy in the upcoming inspection. 

**Target variable**: Health status (Healthy)

**Descriptive Features:** Past Health Descriptors and Weather trends

**Models:** Logistic Regression

In [106]:
# Import standard libraries
import pandas as pd
import pickle

# Imports from sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, roc_auc_score, make_scorer, accuracy_score
from sklearn.preprocessing import MinMaxScaler

In [ ]:
try:
    # 1. Load Data
    print("Loading training set...")
    path_to_data = '../data/test_train/training_data.pkl'
    with open(path_to_data, 'rb') as f:
        loaded_data = pickle.load(f)

    X_train = loaded_data["X_train"]
    groups_train = loaded_data["groups_train"]
    Y_train = loaded_data["Y_train"]

    # 2. Define Target and Features
    target = 'Healthy'
    group_key = 'HiveID'

    # 2.1 Historical and Weather Features
    health_history = [
        'Is_First_Inspection', 'Days_Since_Last_Inspection', 'Hive_Age_Days',
        'Prev_Brood_Status', 'Prev_Bees_Status', 'Prev_Queen_Status',
        'Prev_Food_Status', 'Prev_Stressors_Status', 'Prev_Space_Status']
    weather_features = [
        'Avg_prcp', 'Avg_wind', 'Avg_tmax', 'Avg_tmin', 'Avg_tavg',
        'Avg_snow', 'Num_frost_days']

    health_weather_features = health_history + weather_features
    features_to_use = health_weather_features

    print(f"\n--- RUNNING BASELINE ---")
    print(f"Target: {target}")
    print(f"Features used ({len(features_to_use)}): {features_to_use}")

    print("\n--- DATA SHAPEs ---")
    print(f"X shape: {X_train.shape}")
    print(f"Y shape: {Y_train.shape}")

    # 4. Calculate scale_pos_weight for imbalance
    scale_pos_weight = (Y_train == 0).sum() / (Y_train == 1).sum()
    print(f"\nCalculated scale_pos_weight for imbalance: {scale_pos_weight:.2f}")

    # 5. Create Preprocessing Pipeline 
    # 5.1 Since all features are numeric, we just need the imputer. Add scaling as well
    numeric_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value=-1)),
        ('scaler', MinMaxScaler()) 
    ])

    # 5.2 Combine into ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numeric_transformer, features_to_use),
        ],
        remainder='passthrough'
    )

    # 6. Create Model Pipeline
    model = LogisticRegression(random_state=42, max_iter=2000, class_weight='balanced',n_jobs=-1)
    baseline_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # 7. Define Scorers
    scorers = {
        'f1_score': make_scorer(f1_score, pos_label=1),
        'roc_auc': make_scorer(roc_auc_score),
        'accuracy': make_scorer(accuracy_score)
    }
    
    # 8. Fit Model
    results = baseline_pipeline.fit(X_train, Y_train)

    # 9. Print Results
    print("\n--- BASELINE RESULTS ---")
    print("+" + "-"*48 + "+")
    for scorer_name in scorers.keys():
        score = scorers[scorer_name](results, X_train, Y_train)
        print(f"| {scorer_name.ljust(20)} | {score:.4f} {' '*(20 - len(f'{score:.4f}'))}|")
    print("+" + "-"*48 + "+")

except FileNotFoundError:
    print("\n--- ERROR ---")
    print(f"Could not find the file '{path_to_data}'.")
except Exception as e:
    print(f"An error occurred: {e}")

Loading training set...

--- RUNNING BASELINE ---
Target: Healthy
Features used (16): ['Is_First_Inspection', 'Days_Since_Last_Inspection', 'Hive_Age_Days', 'Prev_Brood_Status', 'Prev_Bees_Status', 'Prev_Queen_Status', 'Prev_Food_Status', 'Prev_Stressors_Status', 'Prev_Space_Status', 'Avg_prcp', 'Avg_wind', 'Avg_tmax', 'Avg_tmin', 'Avg_tavg', 'Avg_snow', 'Num_frost_days']

--- DATA SHAPEs ---
X shape: (1596, 16)
Y shape: (1596,)

Calculated scale_pos_weight for imbalance: 1.69

--- BASELINE RESULTS ---
+------------------------------------------------+
| f1_score             | 0.6580               |
| roc_auc              | 0.7207               |
| accuracy             | 0.7187               |
+------------------------------------------------+
